# Kanoniv Playground
## Identity Resolution for AI Agents - Live in 2 Minutes

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kanoniv/kanoniv/blob/main/examples/kanoniv-playground.ipynb)

**What you'll see:** Three AI agents interact with the same person across different channels. Each agent knows her by a different identifier. Kanoniv resolves all three into one canonical identity - deterministically, in real-time.

| Channel | Identifier | What happens |
|---------|-----------|-------------|
| Web browse | Personal Gmail | New entity created |
| Support chat | Work email + phone | **Merged** via Fellegi-Sunter name matching (score 8.5) |
| SMS alerts | Phone only | **Resolved** via handle index in O(1) |
| Different person | Different everything | Correctly kept **separate** |

No installs beyond `httpx`. No local engine. Just API calls to the Kanoniv sandbox.

---

### Install dependencies

Just `httpx` for HTTP calls. Nothing else needed.

In [ ]:
!pip install -q httpx

### Connect to Kanoniv Sandbox

This uses a **sandbox API key** (`kn_test_...`). Sandbox has a 1,000 entity limit, full API access, and instant reset. Your data is isolated and ephemeral.

To get your own sandbox key: `pip install kanoniv` and run `kanoniv auth login`.

In [ ]:
import httpx
import json
from IPython.display import display, Markdown, HTML

API = "https://api.kanoniv.com"
KEY = "kn_test_01eeeaaf4193bb4c4dd3ec01e4f591898d8f0a4f42e79cf30b37e685edae084e"
HEADERS = {"X-API-Key": KEY, "Content-Type": "application/json"}
client = httpx.Client(base_url=API, headers=HEADERS, timeout=15)

def pp(data):
    """Pretty-print JSON with syntax highlighting."""
    formatted = json.dumps(data, indent=2, default=str)
    display(HTML(f'<pre style="background:#1a1a2e;color:#e8e8ed;padding:12px;border-radius:8px;font-size:13px;border:1px solid #333">{formatted}</pre>'))

def badge(text, color):
    """Colored inline badge."""
    colors = {
        'green': ('#34d399', '#065f46'),
        'blue': ('#60a5fa', '#1e3a5f'),
        'purple': ('#a78bfa', '#3b1f6e'),
        'gold': ('#c5a572', '#4a3b1f'),
        'red': ('#f87171', '#5f1e1e'),
    }
    fg, bg = colors.get(color, ('#e8e8ed', '#333'))
    return f'<span style="background:{bg};color:{fg};padding:2px 8px;border-radius:12px;font-size:11px;font-weight:700;letter-spacing:0.5px">{text}</span>'

print("Connected to Kanoniv API")
print(f"Sandbox key: {KEY[:12]}...")

---

## Step 0: Reset Sandbox & Upload Identity Spec

An **identity spec** is the configuration file that tells Kanoniv how to resolve records into entities. Think of it as a schema for identity - it declares:

- **Sources** - where records come from (CRM, support tickets, web events, etc.)
- **Blocking** - how to narrow down candidate pairs efficiently before scoring (avoids comparing every record against every other record)
- **Rules** - deterministic matching rules (exact email match, fuzzy name similarity)
- **Scoring** - probabilistic Fellegi-Sunter weights that produce a composite match score
- **Survivorship** - how to pick the "winning" value when merging conflicting fields
- **Thresholds** - score cutoffs that decide: match, possible match, or no match

One spec, uploaded once. Every `resolve` call after that uses the compiled execution plan.

In [ ]:
SPEC = """
api_version: kanoniv/v2
identity_version: getdupe_shopper_v1

metadata:
  name: GetDupe Shopper Identity
  description: Cross-channel identity resolution for beauty product shoppers
  owner: getdupe
  tags: [commerce, b2c, beauty, cross-channel]

# What we're resolving - one "shopper" entity across all channels
entity:
  name: shopper
  description: Canonical shopper across web, support, SMS, and social channels

# Three data sources feeding into the same entity type.
# Each source maps its columns to canonical attribute names.
sources:
  - name: getdupe_web        # Web sessions
    system: getdupe
    table: sessions
    id: external_id           # Unique record ID within this source
    attributes:
      email: email
      first_name: first_name
      last_name: last_name
      phone: phone

  - name: getdupe_support     # Support tickets
    system: getdupe
    table: tickets
    id: external_id
    attributes:
      email: email
      first_name: first_name
      last_name: last_name
      phone: phone

  - name: getdupe_sms         # SMS opt-ins
    system: getdupe
    table: sms_optins
    id: external_id
    attributes:
      email: email
      first_name: first_name
      last_name: last_name
      phone: phone

# Blocking narrows the search space before scoring.
# Instead of comparing every record to every other record (O(n^2)),
# blocking groups records by shared keys and only scores within groups.
blocking:
  strategy: progressive       # Try cheap keys first, fall back to broader ones
  tiers:
    - keys: [[email], [phone]]         # Tier 1: exact email OR phone match
      candidate_limit: 10
    - keys: [[first_name, last_name]]  # Tier 2: name pair (catches different emails)
      candidate_limit: 30

# Deterministic matching rules (applied before probabilistic scoring)
rules:
  - name: email_exact
    type: exact
    field: email
    weight: 1.0
    options:
      case_insensitive: true
      normalize: true         # Strips whitespace, lowercases

  - name: phone_exact
    type: exact
    field: phone
    weight: 0.9
    options:
      normalize: true         # E.164 normalization (+1-415... -> 14155550199)

  - name: name_similarity
    type: similarity
    field: first_name
    algorithm: jaro_winkler   # Fuzzy string comparison (handles typos, nicknames)
    threshold: 0.85           # Min similarity to count as a match
    weight: 0.6

# When two records merge, which value wins?
survivorship:
  default: most_recent        # Keep the newest value for each field

# Fellegi-Sunter probabilistic scoring configuration.
# Each field gets m (match) and u (unmatch) probabilities:
#   m = P(fields agree | records are the same person)
#   u = P(fields agree | records are different people)
# The log-likelihood ratio log2(m/u) determines each field's contribution.
decision:
  thresholds:
    match: 0.8
    review: 0.5
    reject: 0.2
  scoring:
    strategy: fellegi_sunter
    fields:
      - name: email
        comparator: exact
        weight: 2.0
        m_probability: 0.95   # 95% chance emails match if same person
        u_probability: 0.001  # 0.1% chance emails match if different people
        normalizer: email
        positive_only: true   # Only reward agreement, don't penalize mismatch
                              # (people have multiple emails)

      - name: phone
        comparator: exact
        weight: 2.0
        m_probability: 0.95
        u_probability: 0.001
        normalizer: phone
        positive_only: true   # Same logic - people have multiple phones

      - name: first_name
        comparator: jaro_winkler
        weight: 1.5
        m_probability: 0.85   # 85% chance names match if same person
        u_probability: 0.05   # 5% chance of random name collision
        normalizer: nickname  # Bill -> William, Mike -> Michael, etc.

      - name: last_name
        comparator: jaro_winkler
        weight: 1.5
        m_probability: 0.85
        u_probability: 0.05
        normalizer: generic

    thresholds:
      match: 5.0              # Score >= 5.0 -> auto-merge
      possible: 2.5           # Score 2.5-5.0 -> flag for human review
      non_match: -5.0         # Score < -5.0 -> definitely not the same
"""

# Reset sandbox (clears all entities, handles, evidence)
reset = client.post("/v1/sandbox/reset").json()
display(Markdown(f"Sandbox reset: **{reset['message']}**"))

# Upload and compile the identity spec
spec_resp = client.post("/v1/identity/specs", json={"raw_yaml": SPEC, "compile": True}).json()
display(Markdown(f"Spec uploaded: valid=**{spec_resp['valid']}**, errors={spec_resp['errors']}"))
display(Markdown("---\n*Ready. Run the cells below to see identity resolution in action.*"))

---

## Step 1: Web Browse - Maya signs up with personal email

Maya discovers GetDupe through a Reddit post about affordable skincare. She signs up with her personal Gmail to save product comparisons.

This is the first time Kanoniv sees this person. A new canonical entity is created.

In [ ]:
r1 = client.post("/v1/resolve/realtime", json={
    "source_name": "getdupe_web",
    "external_id": "session_maya_browse_001",
    "data": {
        "email": "maya.skincare@gmail.com",
        "first_name": "Maya",
        "last_name": "Torres"
    }
}).json()

entity_id = r1["entity_id"]

display(HTML(f"""
<div style='background:#1a1a2e;padding:16px 20px;border-radius:10px;border:1px solid #333;margin:8px 0'>
  <div style='font-size:18px;font-weight:700;color:#e8e8ed;margin-bottom:8px'>
    {badge('NEW', 'blue')} Entity created
  </div>
  <div style='font-size:13px;color:#8b8b96;line-height:1.8'>
    Entity ID: <code style='color:#c5a572'>{entity_id}</code><br>
    Source: getdupe_web | Fields: email, first_name, last_name
  </div>
</div>
"""))

pp(r1["canonical_data"])

---

## Step 2: Support Chat - Maya uses her work email

A week later, Maya contacts support using her **work email** (`maya.torres@company.com`). Completely different email address. She also provides her phone number.

Kanoniv runs Fellegi-Sunter probabilistic scoring:
- `email`: different (positive_only, so no penalty)
- `first_name`: "Maya" = "Maya" (Jaro-Winkler 1.0, contribution +4.25)
- `last_name`: "Torres" = "Torres" (Jaro-Winkler 1.0, contribution +4.25)
- **Total score: ~8.5** (well above 5.0 threshold)

**This is the key moment. Two different emails, same person. One API call.**

In [ ]:
r2 = client.post("/v1/resolve/realtime", json={
    "source_name": "getdupe_support",
    "external_id": "ticket_maya_niacinamide_002",
    "data": {
        "email": "maya.torres@company.com",
        "first_name": "Maya",
        "last_name": "Torres",
        "phone": "+14155550199"
    }
}).json()

merged = r2["entity_id"] == entity_id

display(HTML(f"""
<div style='background:#1a1a2e;padding:16px 20px;border-radius:10px;border:1px solid {'#065f46' if merged else '#5f1e1e'};border-left:3px solid {'#34d399' if merged else '#f87171'};margin:8px 0'>
  <div style='font-size:18px;font-weight:700;color:#e8e8ed;margin-bottom:8px'>
    {badge('MERGED', 'green')} Same entity as web browse? <span style='color:{'#34d399' if merged else '#f87171'};font-size:20px'>{'Yes' if merged else 'No'}</span>
  </div>
  <div style='font-size:13px;color:#8b8b96;line-height:1.8'>
    FS Score: <strong style='color:#c5a572'>{r2['confidence']:.2f}</strong> (threshold: 5.0)<br>
    Resolution: <strong>{r2['resolution']}</strong> | New handles registered: work email + phone
  </div>
</div>
"""))

pp(r2["canonical_data"])

---

## Step 3: SMS Alerts - Phone number only

Maya opts into SMS price alerts. **Only her phone number is provided.** No name. No email.

Kanoniv resolves her instantly through the **handle index** - the phone `+14155550199` was auto-registered when it appeared in the support chat. O(1) Redis lookup. No scoring needed.

In [ ]:
r3 = client.post("/v1/resolve/realtime", json={
    "source_name": "getdupe_sms",
    "external_id": "sms_maya_pricedrop_003",
    "data": {
        "phone": "+14155550199"
    }
}).json()

same = r3["entity_id"] == entity_id

display(HTML(f"""
<div style='background:#1a1a2e;padding:16px 20px;border-radius:10px;border:1px solid #3b1f6e;border-left:3px solid #a78bfa;margin:8px 0'>
  <div style='font-size:18px;font-weight:700;color:#e8e8ed;margin-bottom:8px'>
    {badge('HANDLE HIT', 'purple')} Resolved via handle index
  </div>
  <div style='font-size:13px;color:#8b8b96;line-height:1.8'>
    Phone <code style='color:#a78bfa'>+14155550199</code> already registered from support chat<br>
    Same entity as web browse? <strong style='color:{'#34d399' if same else '#f87171'}'>{'Yes' if same else 'No'}</strong> | No scoring needed - O(1) lookup
  </div>
</div>
"""))

---

## Step 4: Different Person - Precision Check

Lina Park is a completely different shopper. Different name, different email, different phone.

Kanoniv correctly creates a **separate entity**. The system isn't blindly merging everything - it has both recall (merging Maya across channels) and precision (keeping Lina separate).

In [ ]:
r4 = client.post("/v1/resolve/realtime", json={
    "source_name": "getdupe_web",
    "external_id": "session_lina_retinol_004",
    "data": {
        "email": "lina.park.derm@gmail.com",
        "first_name": "Lina",
        "last_name": "Park",
        "phone": "+12125550742"
    }
}).json()

lina_id = r4["entity_id"]
separate = lina_id != entity_id

display(HTML(f"""
<div style='background:#1a1a2e;padding:16px 20px;border-radius:10px;border:1px solid #333;margin:8px 0'>
  <div style='font-size:18px;font-weight:700;color:#e8e8ed;margin-bottom:8px'>
    {badge('NEW', 'blue')} Separate entity created
  </div>
  <div style='font-size:13px;color:#8b8b96;line-height:1.8'>
    Entity ID: <code style='color:#c5a572'>{lina_id}</code><br>
    Same as Maya? <strong style='color:{'#34d399' if separate else '#f87171'}'>{'No (correct)' if separate else 'Yes (wrong!)'}</strong>
  </div>
</div>
"""))

---

## Step 5: Cross-Channel Handles

Every email and phone seen during resolve is auto-registered as a **handle**. Any downstream agent can look up Maya by any of her identifiers - sub-millisecond, via Redis.

In [ ]:
import time; time.sleep(1)  # let async handle writes land

handles = client.get(f"/v1/entities/{entity_id}/handles").json()

rows = ""
for h in handles:
    color = '#60a5fa' if h['handle_type'] == 'email' else '#34d399'
    rows += f"<tr><td style='color:{color};font-weight:600'>{h['handle_type']}</td><td><code style='color:#e8e8ed'>{h['handle_value']}</code></td><td style='color:#8b8b96'>{h['source']}</td></tr>"

display(HTML(f"""
<div style='background:#1a1a2e;padding:16px 20px;border-radius:10px;border:1px solid #333;margin:8px 0'>
  <div style='font-size:15px;font-weight:700;color:#e8e8ed;margin-bottom:12px'>
    {len(handles)} handles for Maya Torres
  </div>
  <table style='width:100%;border-collapse:collapse;font-size:13px'>
    <tr style='border-bottom:1px solid #333'>
      <th style='text-align:left;padding:6px 12px 6px 0;color:#55555f;font-size:10px;text-transform:uppercase;letter-spacing:1px'>Type</th>
      <th style='text-align:left;padding:6px 12px 6px 0;color:#55555f;font-size:10px;text-transform:uppercase;letter-spacing:1px'>Value</th>
      <th style='text-align:left;padding:6px 0;color:#55555f;font-size:10px;text-transform:uppercase;letter-spacing:1px'>Registered By</th>
    </tr>
    {rows}
  </table>
</div>
"""))

# Prove any identifier resolves to the same person
display(Markdown("**Handle lookups** - any agent finds Maya by any identifier:\n"))
for handle in ["email:maya.skincare@gmail.com", "email:maya.torres@company.com", "phone:+14155550199"]:
    lk = client.post("/v1/resolve/handle", json={"handle": handle}).json()
    name = f"{lk['canonical_data'].get('first_name', '')} {lk['canonical_data'].get('last_name', '')}"
    print(f"  {handle:<38} -> {name} ({lk['entity_id'][:12]}...)")

---

## Step 6: Evidence Timeline

Every resolve writes an evidence row: the decision, the score, the raw input, and per-field Fellegi-Sunter signal breakdown. This is the identity **flight recorder** - full auditability.

In [ ]:
evidence = client.get(f"/v1/entities/{entity_id}/evidence").json()

display(Markdown(f"### {len(evidence)} resolve events for Maya Torres\n"))

for e in evidence:
    fields = list(e["raw_input"].keys())
    decision = e['decision']
    score = e['composite_score']
    source = e['source_name']

    print(f"  {decision:>10} | score: {score:>5.1f} | source: {source:<18} | fields: {', '.join(fields)}")

    # Show per-field FS signals when available
    if e.get("signals") and e["signals"].get("field_scores"):
        for fs in e["signals"]["field_scores"]:
            pos = " (positive_only)" if fs.get("positive_only") else ""
            print(f"               {fs['field']:>12}: contribution={fs['contribution']:>+6.2f}  agreement={fs.get('agreement', 'N/A')}{pos}")

print()

# Evidence summary for both shoppers
display(Markdown("### Evidence Summary\n"))
for name, eid in [("Maya Torres", entity_id), ("Lina Park", lina_id)]:
    s = client.get(f"/v1/entities/{eid}/evidence/summary").json()
    print(f"  {name}: {s['total_resolves']} resolves | {s['unique_sources']} sources | decisions: {json.dumps(s['decisions'])}")

---

## What Just Happened

In 4 API calls, Kanoniv:

1. **Created** a canonical identity from a web signup
2. **Merged** a support ticket with a different email - using Fellegi-Sunter name matching (score 8.5)
3. **Resolved** a phone-only SMS record in O(1) via the handle index
4. **Separated** a different person correctly

Every decision is deterministic, explainable, and auditable.

| What | How |
|------|-----|
| 3 channels | Web, Support, SMS |
| 2 emails | Personal Gmail + work email |
| 1 phone | Auto-registered as handle |
| **1 canonical Maya Torres** | Deterministic identity |
| **1 separate Lina Park** | Precision, not just recall |

---

## Try It Yourself

```bash
# Install the SDK
pip install kanoniv

# Get a sandbox API key
kanoniv auth login

# Or use MCP with Claude/Cursor
npx @kanoniv/mcp
```

**Links:**
- Documentation: [docs.kanoniv.com](https://docs.kanoniv.com)
- Python SDK: [pypi.org/project/kanoniv](https://pypi.org/project/kanoniv)
- MCP Server: [npmjs.com/package/@kanoniv/mcp](https://npmjs.com/package/@kanoniv/mcp)
- GitHub: [github.com/kanoniv/kanoniv](https://github.com/kanoniv/kanoniv)